# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [2]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Configuración según la imagen (filas 0..4, cols 0..5)
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # terminales con recompensa
        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # carga
            (3, 5): -10.0,  # peligro mortal
        }

        # peligros no terminales
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # probabilities: (main, left, right)
        self.normal_probs = (0.9, 0.05, 0.05)
        self.slippery_probs = (0.6, 0.2, 0.2)

        # acciones: UP, DOWN, LEFT, RIGHT
        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        S = []
        for r in range(self.height):
            for c in range(self.width):
                s = (r, c)
                if self.is_valid_state(s):
                    S.append(s)
        return S

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        # transitable normal
        return self.living_reward

    def _move(self, state, action):
        """Intenta mover; si choca con pared o borde, queda en el mismo estado."""
        r, c = state
        dr, dc = action
        ns = (r + dr, c + dc)
        if not self.is_valid_state(ns):
            return state
        return ns

    def get_transition_probs(self, state, action):
        """
        Devuelve lista de (next_state, prob)
        """
        # Si es terminal, permanece
        if self.is_terminal(state):
            return [(state, 1.0)]

        # elegir probabilidades según tipo de piso
        if state in self.slippery_states:
            main_p, left_p, right_p = self.slippery_probs
        else:
            main_p, left_p, right_p = self.normal_probs

        # Determinar left/right desviaciones según la acción
        # mapping: for UP, left=LEFT, right=RIGHT
        # for DOWN, left=RIGHT, right=LEFT
        # for LEFT, left=DOWN, right=UP
        # for RIGHT, left=UP, right=DOWN
        a = action
        if a == (-1, 0):  # UP
            left_a = (0, -1)
            right_a = (0, 1)
        elif a == (1, 0):  # DOWN
            left_a = (0, 1)
            right_a = (0, -1)
        elif a == (0, -1):  # LEFT
            left_a = (1, 0)
            right_a = (-1, 0)
        else:  # RIGHT
            left_a = (-1, 0)
            right_a = (1, 0)

        # posibles movimientos
        main_ns = self._move(state, a)
        left_ns = self._move(state, left_a)
        right_ns = self._move(state, right_a)

        probs = {}
        probs[main_ns] = probs.get(main_ns, 0.0) + main_p
        probs[left_ns] = probs.get(left_ns, 0.0) + left_p
        probs[right_ns] = probs.get(right_ns, 0.0) + right_p

        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [3]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [5]:
def expected_next_value(grid, state, action, V):
    transitions = grid.get_transition_probs(state, action)
    return sum(p * V[next_state] for next_state, p in transitions)


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for it in range(1, max_iter+1):
        delta = 0.0
        V_new = V.copy()
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue

            # Bellman optimality
            action_values = []
            for a in grid.actions:
                ev = expected_next_value(grid, s, a, V)
                action_values.append(grid.get_reward(s) + grid.gamma * ev)

            best = max(action_values)
            delta = max(delta, abs(best - V[s]))
            V_new[s] = best

        V = V_new
        if delta < threshold:
            return V, it

    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = (0,0)
            continue
        best_a = None
        best_val = -float('inf')
        for a in grid.actions:
            ev = expected_next_value(grid, s, a, V)
            val = grid.get_reward(s) + grid.gamma * ev
            if val > best_val:
                best_val = val
                best_a = a
        policy[s] = best_a
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [6]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for it in range(1, max_iter+1):
        delta = 0.0
        V_new = V.copy()
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue
            a = policy[s]
            ev = expected_next_value(grid, s, a, V)
            v_new = grid.get_reward(s) + grid.gamma * ev
            delta = max(delta, abs(v_new - V[s]))
            V_new[s] = v_new
        V = V_new
        if delta < threshold:
            return V
    return V


def policy_improvement(grid, V):
    policy = {}
    stable = True
    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = (0,0)
            continue
        best_a = None
        best_val = -float('inf')
        for a in grid.actions:
            ev = expected_next_value(grid, s, a, V)
            val = grid.get_reward(s) + grid.gamma * ev
            if val > best_val:
                best_val = val
                best_a = a
        policy[s] = best_a
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # inicializar política arbitraria (p.ej. UP)
    policy = {s: grid.actions[0] for s in grid.states()}

    history = []
    for i in range(max_iter):
        V = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)
        history.append(new_policy)
        # comparar
        changed = any(new_policy[s] != policy[s] for s in grid.states())
        policy = new_policy
        if not changed:
            return policy, V, history
    return policy, V, history



## Parte 4 — Visualización y comparación


In [8]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [9]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{(0, 0): (-1, 0), (0, 1): (-1, 0), (0, 2): (1, 0), (0, 4): (0, 1), (0, 5): (0, 0), (1, 0): (1, 0), (1, 2): (1, 0), (1, 3): (0, 1), (1, 4): (0, 1), (1, 5): (-1, 0), (2, 0): (0, 1), (2, 1): (0, 1), (2, 2): (0, 0), (2, 3): (0, -1), (2, 5): (-1, 0), (3, 0): (0, 1), (3, 1): (0, 1), (3, 2): (-1, 0), (3, 3): (0, -1), (3, 4): (0, -1), (3, 5): (0, 0), (4, 0): (0, 1), (4, 1): (-1, 0), (4, 3): (-1, 0), (4, 4): (0, -1), (4, 5): (0, -1)}, {(0, 0): 


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [10]:
# Experimentos A/B/C

def simulate_policy(grid, policy, start=None, max_steps=100):
    s = start if start is not None else grid.start
    traj = [s]
    for _ in range(max_steps):
        if grid.is_terminal(s):
            break
        a = policy[s]
        transitions = grid.get_transition_probs(s, a)
        # sample deterministically the most probable next state for clarity
        next_state = max(transitions, key=lambda x: x[1])[0]
        traj.append(next_state)
        s = next_state
    return traj

print("-- Experimento A: living_reward = -0.1 --")
gridA = WarehouseMDP()
gridA.living_reward = -0.1
V_A, itA = value_iteration(gridA)
pi_A = extract_policy(gridA, V_A)
print("Iteraciones VI:", itA)
print("Acción desde START:", pi_A[gridA.start])
print("Trayectoria ejemplo:", simulate_policy(gridA, pi_A, gridA.start))
print()

print("-- Experimento B: piso resbaloso más inestable (main 0.4) --")
gridB = WarehouseMDP()
gridB.slippery_probs = (0.4, 0.3, 0.3)
V_B, itB = value_iteration(gridB)
pi_B = extract_policy(gridB, V_B)
print("Iteraciones VI:", itB)
print("Acción desde START:", pi_B[gridB.start])
print("Trayectoria ejemplo:", simulate_policy(gridB, pi_B, gridB.start))
print()

print("-- Experimento C: gamma = 0.99 --")
gridC = WarehouseMDP()
gridC.gamma = 0.99
V_C, itC = value_iteration(gridC)
pi_C = extract_policy(gridC, V_C)
print("Iteraciones VI:", itC)
print("Acción desde START:", pi_C[gridC.start])
print("Trayectoria ejemplo:", simulate_policy(gridC, pi_C, gridC.start))


-- Experimento A: living_reward = -0.1 --
Iteraciones VI: 26
Acción desde START: (0, 1)
Trayectoria ejemplo: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]

-- Experimento B: piso resbaloso más inestable (main 0.4) --
Iteraciones VI: 22
Acción desde START: (0, 1)
Trayectoria ejemplo: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]

-- Experimento C: gamma = 0.99 --
Iteraciones VI: 24
Acción desde START: (0, 1)
Trayectoria ejemplo: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
